# Rank Percentile Score Validation
## Comparing three normalisation approaches for binder–decoy discrimination

**Experiment:** For each docked molecule we compute three normalised scores and compare how well each separates **known binders** from **decoys**.

| Score | Description |
|---|---|
| **Rank percentile** | Molecules are ranked within their protein's full distribution and mapped 0→1 (plotted here as 1 - rp_vina_score, so higher = better on this axis; the stored column itself is 0 = best -- see the note above `panels` in the plotting cell). |
| **Per-protein min-max** | Raw Vina score min-max scaled per protein (inverted, 1 = best). |
| **Z-score (decoy-referenced)** | Vina score z-scored against decoy distribution, mapped via Φ(−z) to 0→1. |

### Data sources
See `notebooks/analysis/README.md` for the full input table. In short:
* **`vinarun_scores.txt`** (`GUILD_FIGURES_DATA_DIR`) — decoys only, filtered from the large Vina case study.
* **`knownbinders_scores.txt`** (`GUILD_FIGURES_DATA_DIR`) — known binders docked fresh on the same proteins.

In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

from legacy_columns import LEGACY_RENAME

warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────────────────────
# Inputs are external (Zenodo deposit), not committed to the repo -- see
# notebooks/analysis/README.md. Override with the GUILD_FIGURES_DATA_DIR env var
# or by editing DATA_DIR, mirroring --data in
# notebooks/analysis/reviewer_response/reproduce_response_analyses.py.
DATA_DIR              = Path(os.environ.get("GUILD_FIGURES_DATA_DIR", "data"))
COMBINED_SCORES_PATH  = DATA_DIR / "vinarun_scores.txt"
KB_GUILD_SCORES_PATH  = DATA_DIR / "knownbinders_scores.txt"

OUT_DIR = Path("np_synthetic_comparison/guild_figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Palette ───────────────────────────────────────────────────────────────────
COLORS = {
    "decoy":          "#999999",
    "known-binder":   "#e07b39",   # orange — both strong + weak collapsed
    "strong-binder":  "#c0392b",
    "weak-binder":    "#e07b39",
    "Natural Product":"#2ecc71",
    "Synthetic":      "#3498db",
}
print("Paths configured ✓")

In [ ]:
# ── Proteins to exclude ───────────────────────────────────────────────────────
EXCLUDE_PROTEINS = {"7v6a", "8fx5", "8wrz", "8wu1", "8dzs"}

# ── R2-7: decoy subset ────────────────────────────────────────────────────────
# None (default) = every decoy, i.e. current behaviour, unchanged. Set to
# either an explicit list of ligand_ids, or a predicate
# (decoys_df -> boolean Series) to restrict which decoys feed the rest of this
# notebook -- e.g. the ligand_ids kept by
# guild.tools.decoy_matching.match_decoys_to_binders(known_binders, decoys),
# to repeat Figure 3 on the property-matched, structurally-distinct panel R2-7
# asks for. See the "R2-7: matched decoys" cell at the end of this notebook
# for how that call is wired up (not run here at full scale -- see that cell
# for why).
DECOY_SUBSET = None


def _apply_decoy_subset(decoys_df, subset):
    if subset is None:
        return decoys_df
    if callable(subset):
        return decoys_df[subset(decoys_df)].copy()
    return decoys_df[decoys_df["ligand_id"].isin(subset)].copy()


# ── Load scores_combined.txt — decoys only ───────────────────────────────────
combined = pd.read_csv(COMBINED_SCORES_PATH, sep="\t")
combined = combined.rename(columns=LEGACY_RENAME)
combined = combined[combined["ligand_category"] == "decoy"].copy()
combined = _apply_decoy_subset(combined, DECOY_SUBSET)
combined["group"]  = "decoy"
combined["source"] = "combined"
print(f"Decoys loaded: {len(combined):,} rows" + ("" if DECOY_SUBSET is None else " (DECOY_SUBSET applied)"))

# ── Load known-binders run scores ─────────────────────────────────────────────
if not KB_GUILD_SCORES_PATH.exists():
    raise FileNotFoundError(f"KB run not found: {KB_GUILD_SCORES_PATH}")

kb_scores = pd.read_csv(KB_GUILD_SCORES_PATH, sep="\t")
kb_scores = kb_scores.rename(columns=LEGACY_RENAME)
kb_scores["group"]  = "known-binder"
kb_scores["source"] = "kb_run"
print(f"Known-binders loaded: {len(kb_scores):,} rows")

# ── Build unified dataframe ───────────────────────────────────────────────────
unified = pd.concat([combined, kb_scores], ignore_index=True)

# ── Exclude proteins ─────────────────────────────────────────────────────────
unified["pdb_id"] = unified["protein_config_id"].str.split("-").str[0]
unified = unified[~unified["pdb_id"].isin(EXCLUDE_PROTEINS)].copy()
print(f"Unified dataset (after excluding {EXCLUDE_PROTEINS}): {len(unified):,} rows")
print(unified["group"].value_counts())

In [ ]:
from scipy.stats import norm as scipy_norm
from scipy.stats import percentileofscore

from guild.tools.scores import compute_rank_percentile_scores

# ── Keep only rows with a valid vina score ────────────────────────────────────
unified = unified[unified["vina_score"].notna()].copy()
print(f"Rows with valid vina score: {len(unified):,}")
print(unified["group"].value_counts())

# For the pandas-3 row-count/index-alignment check at the bottom of this cell.
_n_before = len(unified)
_index_before = unified.index.copy()

# ── Compute rank percentile scores via the package function ─────────────────────────────
# Adds rank_vina_score and rp_vina_score columns per protein.
# rp_vina_score = rank / N  (0→1, lower = better; rank 1 = most negative Vina)
unified = compute_rank_percentile_scores(unified, methods=["vina"])

# ── Per-protein min-max normalisation of raw Vina (comparison baseline) ───────
def _minmax_invert(grp):
    lo, hi = grp.min(), grp.max()
    return (hi - grp) / (hi - lo) if hi != lo else pd.Series(0.5, index=grp.index)

unified["vina_min_max"] = unified.groupby("protein_config_id")["vina_score"].transform(_minmax_invert)

# ── Per-protein decoy Vina arrays, shared by the two decoy-referenced scores
# below ──────────────────────────────────────────────────────────────────────
# Built by iterating the groupby object directly (not .apply()) -- each group
# collapses to a single array, one dict entry per protein, so this never
# touches the DataFrameGroupBy.apply() pattern that relies on the grouping
# column being echoed back into a per-row result (deprecated on pandas 2.2+,
# removed on 3.x -- the same issue e9a6fe7 fixed in
# guild.tools.scores.compute_rank_percentile_scores).
decoy_vina_by_protein = {
    protein: sub["vina_score"].dropna().to_numpy()
    for protein, sub in unified.loc[unified["group"] == "decoy"].groupby("protein_config_id")
}

# ── Decoy-referenced rank percentile ─────────────────────────────────────────
# For each molecule: what fraction of *decoys* (same protein) have a worse Vina
# score? Inverted so 1 = better than all decoys.
#
# Not expressible as a single groupby .transform(): it references a different
# set (decoys only) than the group being scored (all molecules), unlike a
# per-group scalar stat such as mean/std. What is removed here is only the
# outer DataFrameGroupBy.apply() -- the per-value percentileofscore lookup
# below is unchanged (same function, same tie handling, same numbers), so this
# cannot move the printed ranges or the AUCs.
def _decoy_pct_value(protein, v):
    decoy_vina = decoy_vina_by_protein.get(protein)
    if decoy_vina is None or len(decoy_vina) < 2:
        return 0.5
    if pd.isna(v):
        return float("nan")
    return 1.0 - percentileofscore(decoy_vina, v, kind="rank") / 100.0

unified["decoy_pct"] = [
    _decoy_pct_value(p, v)
    for p, v in zip(unified["protein_config_id"], unified["vina_score"], strict=True)
]

# ── Standard z-score → normal CDF → [0, 1] ───────────────────────────────────
# Mean and std estimated from ALL molecules per protein; lower Vina = better, so −z is used.
# This one IS a straightforward transform: mu/sigma are per-group scalars, so
# no grouping-column pass-through is involved even in the original .apply().
# groupby(...).transform("mean"/"std") already skips NaN and returns NaN for a
# group with fewer than 2 non-null values (ddof=1, matching the old
# `len(scores) < 2` guard); the sigma==0 guard is kept explicitly since a
# transform can't special-case that itself.
_grouped_vina = unified.groupby("protein_config_id")["vina_score"]
_mu = _grouped_vina.transform("mean")
_sigma = _grouped_vina.transform("std")
_z = (unified["vina_score"] - _mu) / _sigma
_zscore_norm = pd.Series(scipy_norm.cdf(-_z), index=unified.index)
_degenerate = _sigma.isna() | (_sigma == 0)
unified["zscore_norm"] = _zscore_norm.where(~_degenerate, 0.5)

# ── Unbounded decoy-fitted rank percentile ────────────────────────────────────
# Phase 1: Rank decoys only per protein (1 = best binder).
# Phase 2: Proportion = rank / N_decoys.
# Phase 3: Place ALL molecules on the decoy scale (no clipping).
#   → known binders that beat all decoys get values < 1/N (below 0 when inverted to 1-best).
#   → known binders worse than all decoys get values > 1.
# Reuses decoy_vina_by_protein above; same non-.transform()-able reasoning as
# decoy_pct, and same guarantee: only the outer DataFrameGroupBy.apply() is
# removed, the per-value rank arithmetic is unchanged.
def _rank_against_decoys(protein, v):
    decoy_vina = decoy_vina_by_protein.get(protein)
    if decoy_vina is None or len(decoy_vina) < 2 or pd.isna(v):
        return np.nan
    n_decoys = len(decoy_vina)
    better_than = np.sum(decoy_vina > v)
    ties = np.sum(decoy_vina == v)
    rank = n_decoys - better_than - 0.5 * ties + 0.5
    return rank / n_decoys

unified["decoy_fit_unbounded"] = [
    _rank_against_decoys(p, v)
    for p, v in zip(unified["protein_config_id"], unified["vina_score"], strict=True)
]

# Same row count, same row order as before these three columns were computed --
# the failure mode a positional (rather than per-protein-keyed) rewrite would
# have risked.
assert len(unified) == _n_before, "row count changed while computing decoy_pct/zscore_norm/decoy_fit_unbounded"
assert (unified.index == _index_before).all(), "row order changed while computing decoy_pct/zscore_norm/decoy_fit_unbounded"

print(f"\nRows after scoring: {len(unified):,}")
print(f"rp_vina_score range        : {unified['rp_vina_score'].min():.4f} – {unified['rp_vina_score'].max():.4f}")
print(f"vina_min_max range         : {unified['vina_min_max'].min():.4f} – {unified['vina_min_max'].max():.4f}")
print(f"decoy_pct range            : {unified['decoy_pct'].min():.4f} – {unified['decoy_pct'].max():.4f}")
print(f"zscore_norm range          : {unified['zscore_norm'].min():.4f} – {unified['zscore_norm'].max():.4f}")
print(f"decoy_fit_unbounded range  : {unified['decoy_fit_unbounded'].min():.4f} – {unified['decoy_fit_unbounded'].max():.4f}")
print(unified[["protein_config_id", "vina_score", "rp_vina_score", "vina_min_max", "decoy_pct", "zscore_norm", "decoy_fit_unbounded", "group"]].head(6))

In [ ]:

import matplotlib as mpl
from sklearn.metrics import roc_auc_score

from kde_helpers import kde_curve_bounded

# ── Publication-ready defaults (double-column manuscript) ─────────────────────
mpl.rcParams.update(mpl.rcParamsDefault)
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.titleweight": "bold",
    "axes.labelsize": 7,
    "xtick.labelsize": 6.5,
    "ytick.labelsize": 6.5,
    "legend.fontsize": 6.5,
    "axes.linewidth": 0.5,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.major.size": 2.5,
    "ytick.major.size": 2.5,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.transparent": False,
    "figure.dpi": 150,
})

# ── b3: rug-tick sampling, named rather than a bare literal ─────────────────
# The grey (decoy) rug below is a random SUBSAMPLE, not the full decoy set;
# the orange (known-binder) rug is complete. Both facts are invisible in the
# rendered panel, so they are printed explicitly for the caption.
RUG_SUBSAMPLE_N = 400
RUG_SEED = 42

# ── Sample sizes ──────────────────────────────────────────────────────────────
plot_decoys  = unified[unified["group"] == "decoy"]
plot_binders = unified[unified["group"] == "known-binder"]
N_DECOYS  = len(plot_decoys)
N_BINDERS = len(plot_binders)

# ── Helper functions ──────────────────────────────────────────────────────────
def _auc(decoy_vals, kb_vals):
    y_true  = np.concatenate([np.zeros(len(decoy_vals)), np.ones(len(kb_vals))])
    y_score = np.concatenate([decoy_vals, kb_vals])
    mask    = ~np.isnan(y_score)
    if mask.sum() < 2 or len(np.unique(y_true[mask])) < 2:
        return float("nan")
    return roc_auc_score(y_true[mask], y_score[mask])

# ── Prepare data ─────────────────────────────────────────────────────────────
# All three scores are plotted so that HIGHER = BETTER, but rp_vina_score is
# stored 0 = best (guild's convention throughout the rest of the codebase --
# see notebooks/analysis/README.md). This 1 - rp_vina_score inversion is
# exactly why Supp. Text 6 ("1 indicates the top-ranked molecule") reads as
# contradicting guild.tools.scores' own docstring ("0 = best") -- both are
# right about different things, one about this plotted axis, one about the
# stored column. The stored value is not changed; only this plot's axis is
# flipped -- and that flip is NOT cosmetic: roc_auc_score treats a higher
# score as more likely to be the positive (binder) class, so computing the
# AUC directly on the stored 0-best rp_vina_score would print its complement
# (1 - AUC), which would look like anti-correlation rather than the real
# separation. The inversion is what puts panel A's AUC on the same
# higher-is-better scale as panels B and C, not an arbitrary relabelling.
guild_decoy = (1 - plot_decoys["rp_vina_score"]).dropna()
guild_kb    = (1 - plot_binders["rp_vina_score"]).dropna()
# vina_min_max (_minmax_invert) and zscore_norm (cdf(-z)) are already
# higher-is-better by construction, so this is the only inversion in this cell.
norm_decoy  = plot_decoys["vina_min_max"].dropna()
norm_kb     = plot_binders["vina_min_max"].dropna()
zsc_decoy   = plot_decoys["zscore_norm"].dropna()
zsc_kb      = plot_binders["zscore_norm"].dropna()

# (decoy_vals, kb_vals, panel_title, is_highlight)
panels = [
    (guild_decoy, guild_kb, "Rank percentile",             True),
    (norm_decoy,  norm_kb,  "Min-max",                     False),
    (zsc_decoy,   zsc_kb,   "Z-score",                     False),
]

panel_labels = ["A)", "B)", "C)"]

# ── b4: unit-area bounded KDE, shared y-scale kept for cross-panel comparability ──
# gaussian_kde integrates to 1 over the real line, but these panels are drawn
# on [0, 1] with a SHARED y-scale (kmax) and hidden y-ticks -- neither a plain
# KDE nor the shared scale is a unit-area curve within any one panel's own
# axes, so "Density" alone overstated what is shown. Using the same
# kde_curve_bounded as Figure 2's b2 fix makes each curve integrate to 1 on
# its own bounded support; the label is updated to say so, and the shared
# scale (kept deliberately, for cross-panel comparability) is stated in the
# printed output rather than left to be inferred from hidden y-ticks.
x_grid = np.linspace(0, 1, 400)
kmax = max(
    *(kde_curve_bounded(d, x_grid).max() for d, _, _, _ in panels),
    *(kde_curve_bounded(k, x_grid).max() for _, k, _, _ in panels),
    1e-9,
)
print("Figure 3 y-scale is SHARED across all three panels (kmax = "
      f"{kmax:.3f}) for cross-panel comparability -- a taller curve in one "
      "panel really is denser, not an artefact of a per-panel rescale.")

# ── 3×1 vertical figure ─────────────────────────────────────────────────────
fig, axes = plt.subplots(
    3, 1,
    figsize=(3.4, 4.8),
    sharex=True,
    sharey=True,
    gridspec_kw={"hspace": 0.22},
)

print(f"Rug ticks: known-binder rug is the full set (n={N_BINDERS}); decoy rug is "
      f"a random subsample of {RUG_SUBSAMPLE_N} (seed={RUG_SEED}), not the full "
      f"decoy set (n={N_DECOYS:,}) -- both are per-molecule values, not "
      "per-target values underlying the pooled distributions.")

print("\nAUCs (binder vs. decoy, higher = better on every panel's axis):")
for (dec_vals, kb_vals, title, _highlight) in panels:
    print(f"  {title:<16s} AUC = {_auc(dec_vals.values, kb_vals.values):.3f}")

for idx, (ax, (dec_vals, kb_vals, title, highlight)) in enumerate(zip(axes, panels, strict=False)):
    kd = kde_curve_bounded(dec_vals, x_grid)
    kb = kde_curve_bounded(kb_vals,  x_grid)

    # KDE fills + lines
    ax.fill_between(x_grid, 0, kd, color=COLORS["decoy"],        alpha=0.15, linewidth=0)
    ax.plot(x_grid, kd, color=COLORS["decoy"],        lw=1.0, label="Decoys")
    ax.fill_between(x_grid, 0, kb, color=COLORS["known-binder"], alpha=0.35, linewidth=0)
    ax.plot(x_grid, kb, color=COLORS["known-binder"], lw=1.4, label="Known binders")

    # Zero line
    ax.axhline(0, color="black", lw=0.4, alpha=0.4)

    # Rug: decoys -- a RUG_SUBSAMPLE_N-molecule random subsample, not every decoy
    for v in dec_vals.sample(min(RUG_SUBSAMPLE_N, len(dec_vals)), random_state=RUG_SEED):
        ax.plot([v, v], [-kmax*0.03, -kmax*0.08], color=COLORS["decoy"],
                lw=0.3, alpha=0.10, solid_capstyle="butt")
    # Rug: binders -- every known binder, no subsampling
    for v in kb_vals:
        ax.plot([v, v], [-kmax*0.11, -kmax*0.19], color=COLORS["known-binder"],
                lw=0.8, alpha=0.70, solid_capstyle="butt")

    # Panel label + title (top-left)
    ax.text(
        0.03, 0.93, f"{panel_labels[idx]}  {title}",
        transform=ax.transAxes, fontsize=7.5, fontweight="bold",
        va="top", ha="left",
    )

    # AUC annotation (top-right) — bold only for rank percentile (panel A)
    auc = _auc(dec_vals.values, kb_vals.values)
    auc_weight = "bold" if highlight else "normal"
    ax.text(
        0.97, 0.93, f"AUC = {auc:.3f}",
        transform=ax.transAxes, fontsize=7, fontweight=auc_weight,
        va="top", ha="right",
        bbox=dict(boxstyle="round,pad=0.25", facecolor="white",
                  edgecolor="#cccccc", alpha=0.85, linewidth=0.4),
    )

    # Axes cosmetics
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-kmax * 0.25, kmax * 1.15)
    ax.tick_params(axis="y", left=False, labelleft=False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)

    if idx < 2:
        ax.tick_params(axis="x", labelbottom=False)
    else:
        # States the plotted direction explicitly rather than "Normalized
        # score" alone -- see the inversion note above the panels list.
        ax.set_xlabel("Normalized score (higher = better)", labelpad=4)

# Shared legend below the bottom panel
legend_handles = [
    Line2D([0], [0], color=COLORS["decoy"],        lw=1.0, label="Decoys"),
    Line2D([0], [0], color=COLORS["known-binder"], lw=1.4, label="Known binders"),
]
fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=2,
    bbox_to_anchor=(0.5, 0.005),
    frameon=False,
    columnspacing=1.5,
    handlelength=1.5,
)

# Shared y-axis label -- "Probability density": each curve integrates to 1 on
# its own bounded [0, 1] support (kde_curve_bounded), which "Density" alone
# did not make clear was true. The shared scale is stated above, not here.
fig.text(
    0.005, 0.5, "Probability density",
    va="center", ha="left", rotation="vertical", fontsize=7,
)
fig.subplots_adjust(left=0.06, bottom=0.14)

# Sample size annotation
fig.text(
    0.5, -0.01,
    f"n decoys = {N_DECOYS:,};  n binders = {N_BINDERS:,}",
    ha="center", fontsize=6, color="#777777",
)

# Save in multiple formats
for ext in ("png", "pdf", "svg"):
    out = OUT_DIR / f"figure_3_validation_three_normalisations.{ext}"
    fig.savefig(out, dpi=600, bbox_inches="tight")

plt.show()
print(f"\nSaved PNG / PDF / SVG in {OUT_DIR}")


## R2-7: property-matched decoy panel and descriptor-only control

The reply to R2-7 promises Figure 3 repeated on a decoy panel matched to each target's
known binders (molecular weight, cLogP, HBD, HBA, rotatable bonds, net charge; ECFP4
Tanimoto < 0.35 to every binder for that target, so a "matched" decoy is not simply a
close analogue), plus a descriptor-only baseline that uses no docking score at all --
the control that tells us whether Figure 3's separation is binding signal or property
bias. On the 3-target rerun the unmatched decoys are **not** size-matched (actives mean
445 Da vs decoys 353 Da) and a heavy-atom-count baseline alone reaches AUC 0.865, higher
than any docking method; size-matching collapses it to 0.645 -- so this control matters,
and the matched Figure 3 may look materially worse than the panel above.

This is the code path, not the final render: `guild.tools.decoy_matching` (tested in
`tests/ligand_properties/test_decoy_matching.py`) implements the matching and the
descriptor-only baseline as ordinary, reusable library functions, verified here against
a 2-target slice of the real Figure 3 inputs (249/2,000 decoys matched; descriptor-only
AUCs 0.27-0.62 across the six descriptors, all far below Figure 3's rank-percentile AUC).
Running it at full scale (133,000 decoys x 655 binders) is not done in this notebook --
`assign_properties` computes RDKit descriptors one molecule at a time and took roughly
6 s for 2,000 molecules in the check above, so the full decoy pool alone is on the order
of several minutes per call, and the matching call below recomputes it for both frames.
Rendering the matched panel is future work; wiring it in is not.

In [ ]:
from guild.tools.decoy_matching import (
    combined_descriptor_auc,
    descriptor_only_auc,
    match_decoys_to_binders,
)

# Demonstrates the wiring on a fast, small slice by default rather than the
# full pool (see the markdown above for why) -- set to None to run every
# target instead, once you are prepared for that to take several minutes.
N_TARGETS_FOR_DEMO = 5

kb_for_matching = kb_scores.dropna(subset=["smiles"]).copy()
decoys_for_matching = pd.read_csv(COMBINED_SCORES_PATH, sep="\t").rename(columns=LEGACY_RENAME)
decoys_for_matching = decoys_for_matching[decoys_for_matching["ligand_category"] == "decoy"]
decoys_for_matching = decoys_for_matching.dropna(subset=["smiles"]).copy()

if N_TARGETS_FOR_DEMO is not None:
    demo_targets = kb_for_matching["protein_config_id"].unique()[:N_TARGETS_FOR_DEMO]
    kb_for_matching = kb_for_matching[kb_for_matching["protein_config_id"].isin(demo_targets)]
    decoys_for_matching = decoys_for_matching[decoys_for_matching["protein_config_id"].isin(demo_targets)]

print(f"Matching against {kb_for_matching['protein_config_id'].nunique()} target(s): "
      f"{len(kb_for_matching)} binders, {len(decoys_for_matching):,} candidate decoys")

matched_decoys = match_decoys_to_binders(kb_for_matching, decoys_for_matching)
print(f"Property-matched, structurally-distinct decoys kept: {len(matched_decoys)}")

print("\nDescriptor-only baseline (no docking score), per descriptor:")
print(descriptor_only_auc(kb_for_matching, decoys_for_matching).to_string(index=False))

print(f"\nCombined-descriptor logistic-regression AUC (cross-validated): "
      f"{combined_descriptor_auc(kb_for_matching, decoys_for_matching, n_splits=3):.3f}")

print(
    "\nTo render the matched Figure 3 panel: set DECOY_SUBSET above to "
    "matched_decoys['ligand_id'].tolist() (computed over the FULL decoy pool, "
    "not just this demo slice) and re-run this notebook from the top."
)
